In [13]:
import os
import yaml
import sys
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)
import tensorflow as tf
import argparse
from src.madnet import MADNet, colorize_img
from src.preprocessing import StereoDatasetCreator
from src.losses_and_metrics import SSIMLoss
import matplotlib.pyplot as plt
from types import SimpleNamespace
from src.config_preprocessing import normalize_config
# from src.infer_preprocessing import run_infer

# Load configuration from YAML file
config_path = os.path.join(root_path, "configs/infer_config.yaml")
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file not found: {config_path}")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)
args = SimpleNamespace(**cfg)

In [14]:
if args.output_path is None and args.num_adapt != 0:
    raise ValueError("No output_path provided for adaptation."
                        "Either set num_adapt=0 or provide a path to output_path"
                        f"Provided args: output_path: {args.output_path}, num_adapt: {args.num_adapt}")

# Initialise the model
model = MADNet(
    input_shape=(args.height, args.width, 3),
    # weights=args.weights_path,
    # num_adapt_modules=args.num_adapt,
    # mad_mode=args.mad_mode,
    search_range=args.search_range
)
optimizer = tf.keras.optimizers.Adam(learning_rate=args.lr)
model.compile(
    optimizer=optimizer,
    loss=SSIMLoss(),
    metrics=None,
    run_eagerly=False
)
# Get inferencing data
predict_dataset = StereoDatasetCreator(
    left_dir=args.left_dir,
    right_dir=args.right_dir,
    batch_size=args.batch_size,
    height=args.height,
    width=args.width,
    shuffle=False,
    disp_dir=None
)
predict_ds = predict_dataset()
# inference the dataset
disparities = model.predict(predict_ds, steps=args.steps)

2025-09-24 16:21:56.890790: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f79f4003eb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-24 16:21:56.891906: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2025-09-24 16:21:57.158585: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-24 16:21:57.772166: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-09-24 16:22:07.708334: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'loop_rsqrt_fusion_30', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1758723727.740706  338302 device_compiler.h:196] Compiled cluster using XLA!  This line is l

39/39 ━━━━━━━━━━━━━━━━━━━━ 53s 81ms/step


In [15]:
# View disparity predictions
if args.show_pred:
    for i in range(disparities.shape[0]):
        plt.axis("off")
        plt.grid(visible=None)
        disp = tf.expand_dims(disparities[i, :, :, :], axis=0)
        plt.imshow(colorize_img(disp, cmap='jet')[0])
        plt.show()

In [16]:
# save the checkpoint and saved_model if it was updated
if args.num_adapt != 0:
    model.save_weights(args.output_path)